# NB09 — Dihedral Highlight

Generates a 2D structure PNG with rotatable bonds colour-coded by their
conformational flexibility (Δ° range across simulation frames).

**Prerequisites**: `pip install 'mdatools[pocket]'` (rdkit, cairosvg, Pillow)

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
from pathlib import Path
from mdatools.config import AnalysisConfig

cfg = AnalysisConfig(
    ligand_resname = "UNK",
    dt_ns          = 2.0,
)

# Replica roots (same as other notebooks)
REPLICA_ROOTS = [
    Path("../run01"),
    Path("../run02"),
]

# Representative PDB snapshot for 2-D structure drawing
# (solvent-stripped; output of NB03 template selection)
SNAPSHOT_PDB = Path("../snapshots/best_snap_clean.pdb")

# Rotatable bonds to analyse — list of (label, a1, a2, a3, a4) tuples
# where a1–a4 are atom *names* in the ligand residue.
from mdatools.analysis.dihedral import DihedralDef
DIHEDRAL_DEFS: list[DihedralDef] = [
    # Example: ("chi1", "C1", "C2", "C3", "C4")
]

OUTPUT_PNG = Path("../figures/dihedral_highlight.png")
# ────────────────────────────────────────────────────────────────────────────

## 1. Run dihedral analysis

In [ ]:
import pandas as pd
from mdatools.universe import load_and_align
from mdatools.analysis.dihedral import DihedralAnalyzer

dfs = []
for root in REPLICA_ROOTS:
    top  = next(root.glob("*.pdb"))
    traj = next(root.glob("*.xtc"))
    u    = load_and_align(top, traj, cfg)
    ana  = DihedralAnalyzer(cfg, DIHEDRAL_DEFS)
    dfs.append(ana.run(u))

df_all = pd.concat(dfs, ignore_index=True)
df_all.head()

## 2. Compute per-bond range statistics

In [ ]:
from mdatools.plotting.dihedral_highlight import dihedral_range_df

range_df = dihedral_range_df(df_all)
range_df

## 3. Build highlight_bonds list

Map each dihedral label to the two *central* bond atoms (a2–a3) and
choose a colour from the range.

In [ ]:
from mdatools.plotting.dihedral_highlight import range_color

# atom2 / atom3 are the bond-forming atoms for each dihedral
BOND_ATOMS: dict[str, tuple[str, str]] = {
    # "chi1": ("C1", "C2"),
}

highlight_bonds = []
for _, row in range_df.iterrows():
    name = row["dihedral"]
    if name not in BOND_ATOMS:
        continue
    a1, a2 = BOND_ATOMS[name]
    rgb    = range_color(row["range"])
    hex_c  = "#{:02X}{:02X}{:02X}".format(*rgb)
    highlight_bonds.append({
        "atom1": a1,
        "atom2": a2,
        "label": f"Δ{row['range']:.1f}°",
        "color": hex_c,
    })

highlight_bonds

## 4. Generate PNG

In [ ]:
from IPython.display import Image as IPImage
from mdatools.plotting.dihedral_highlight import plot_dihedral_highlight

out = plot_dihedral_highlight(
    pdb_path       = SNAPSHOT_PDB,
    highlight_bonds= highlight_bonds,
    output_png     = OUTPUT_PNG,
    ligand_resname = cfg.ligand_resname,
)
print(f"Saved → {out}")
IPImage(str(out), width=900)